# Sentinel v4 (balanced 1:1, clean text) — Colab run

Approved 2026-10-04 (`docs/SENTINEL_V4_PROPOSAL.md`). Track: REAL PUBLIC DATA. **One run, one design.**

- **Same as v3 except:** 1,551 training scams (1:1 with legit+promo), and MOZ-Smishing as an evaluation-only external test.
- **Verified build:** refuses to train unless the rebuilt corpus is byte-identical to `sentinel/manifests/sentinel_real_v4_balanced.build.json`, including the sampled scam IDs and the MOZ test file.
- **Run the cells one at a time, in order.** Each step stops on any non-zero exit.
- **Order:** smoke test → full training → **secure the adapter** (zip + checksum + download) → scoring → evaluation → results zip.
- **Acceptance is not run here:** the frozen baseline is scored on MOZ afterwards, outside Colab, and `check_acceptance.py` runs once with both sides complete.

In [ ]:
import subprocess, sys
def sh(cmd):
    print('$', cmd, flush=True)
    r = subprocess.run(cmd, shell=True)
    if r.returncode != 0:
        raise SystemExit(f'STOPPED: command failed with exit code {r.returncode}: {cmd}')

In [ ]:
sh('nvidia-smi')
sh('git clone https://github.com/TanviSaraswat8/Chakravyuh.git')
import os; os.chdir('Chakravyuh')
sh('git log -1 --oneline')

In [ ]:
sh('pip -q install unsloth trl datasets')
sh('pip -q install -r backend/requirements.txt -r backend/requirements-data.txt')

## 1. Datasets (pinned commits / revision, SHA-256 checked)

In [ ]:
sh('python scripts/fetch_real_datasets.py fetch imc25_smishing sp24_gateway_phishing india_spam_sms_junioralive')
sh('python scripts/fetch_real_datasets.py fetch moz_smishing')   # Hugging Face commit 1092f9d9, SHA-256 814a11d9…

## 2. Official UCI zip (upload `sms+spam+collection.zip`)

In [ ]:
from google.colab import files
up = files.upload()
import hashlib, pathlib
z = pathlib.Path(next(iter(up)))
assert hashlib.sha256(z.read_bytes()).hexdigest() == '1587ea43e58e82b14ff1f5425c88e17f8496bfcdb67a583dbff9eefaf9963ce3', 'not the registered UCI zip'
sh(f'mkdir -p data/incoming && mv "{z}" data/incoming/smsspamcollection.zip')
sh('python scripts/register_official_uci.py data/incoming/smsspamcollection.zip')

## 3. Validate, splits, UCI calibration split, v4 build and the approval gate

In [ ]:
sh('python scripts/validate_real_dataset.py imc25_smishing sp24_gateway_phishing india_spam_sms_junioralive uci_sms_spam moz_smishing')
sh('python scripts/prepare_real_splits.py')
sh('python scripts/run_cross_source.py save-uci-split')
sh('python sentinel/build_sft_real.py --config sentinel/configs/sentinel_real_v4_balanced.json')
sh('python sentinel/verify_build.py sentinel_real_v4_balanced')   # must print BUILD MATCHES APPROVED MANIFEST

## 4. SMOKE TEST — 50 messages, 5 steps. Stop here if anything fails.

In [ ]:
sh('python sentinel/finetune.py --base unsloth/Qwen2.5-1.5B-Instruct-bnb-4bit --data sentinel/sft_real/sentinel_real_v4_balanced --out sentinel/outputs/sentinel_real_v4_balanced_smoke --max-seq 384 --epochs 1 --lr 2e-4 --rank 16 --batch 8 --grad-accum 2 --seed 13 --limit 50 --max-steps 5')
sh('python sentinel/predict_hf.py --model sentinel/outputs/sentinel_real_v4_balanced_smoke/lora --data sentinel/sft_real/sentinel_real_v4_balanced --out sentinel/outputs/sentinel_real_v4_balanced_smoke/predictions --limit 50 --latency-n 20')
sh('python sentinel/smoke_check.py --run sentinel/outputs/sentinel_real_v4_balanced_smoke --data sentinel/sft_real/sentinel_real_v4_balanced --n 50')   # must print SMOKE TEST PASSED

## 5. Full training (approved hyperparameters, unchanged)

In [ ]:
sh('mkdir -p sentinel/outputs/sentinel_real_v4_balanced && nvidia-smi > sentinel/outputs/sentinel_real_v4_balanced/hardware.txt && nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv >> sentinel/outputs/sentinel_real_v4_balanced/hardware.txt')
sh('python sentinel/finetune.py --base unsloth/Qwen2.5-1.5B-Instruct-bnb-4bit --data sentinel/sft_real/sentinel_real_v4_balanced --out sentinel/outputs/sentinel_real_v4_balanced --max-seq 384 --epochs 1 --lr 2e-4 --rank 16 --batch 8 --grad-accum 2 --seed 13')

## 6. SECURE THE ADAPTER — before any scoring

In [ ]:
sh('ls -l sentinel/outputs/sentinel_real_v4_balanced/lora && test -s sentinel/outputs/sentinel_real_v4_balanced/lora/adapter_model.safetensors && test -s sentinel/outputs/sentinel_real_v4_balanced/lora/adapter_config.json')
sh('cd sentinel/outputs/sentinel_real_v4_balanced/lora && sha256sum * > ../lora_SHA256SUMS.txt && cat ../lora_SHA256SUMS.txt')
sh('cd sentinel/outputs && zip -qr sentinel_real_v4_balanced_adapter.zip sentinel_real_v4_balanced/lora sentinel_real_v4_balanced/lora_SHA256SUMS.txt sentinel_real_v4_balanced/training_log.json && unzip -tq sentinel_real_v4_balanced_adapter.zip && sha256sum sentinel_real_v4_balanced_adapter.zip')
from google.colab import files
files.download('sentinel/outputs/sentinel_real_v4_balanced_adapter.zip')   # confirm it is saved before running step 7

## 7. Score every test file (full-length prompts, no truncation) and evaluate per source

In [ ]:
sh('python sentinel/predict_hf.py --model sentinel/outputs/sentinel_real_v4_balanced/lora --data sentinel/sft_real/sentinel_real_v4_balanced --out sentinel/outputs/sentinel_real_v4_balanced/predictions')
sh('python sentinel/evaluate_cross_source.py --data sentinel/sft_real/sentinel_real_v4_balanced --pred sentinel/outputs/sentinel_real_v4_balanced/predictions --out sentinel/outputs/sentinel_real_v4_balanced/metrics_real.json')

## 8. Manifest and results download (no weights)

In [ ]:
sh('python sentinel/write_model_manifest.py --run sentinel/outputs/sentinel_real_v4_balanced --data sentinel/sft_real/sentinel_real_v4_balanced --config sentinel/configs/sentinel_real_v4_balanced.json')
sh('cp sentinel/sft_real/sentinel_real_v4_balanced/build_manifest.json sentinel/outputs/sentinel_real_v4_balanced/build_manifest.json && cp sentinel/sft_real/sentinel_real_v4_balanced/sampled_scam_ids.json sentinel/outputs/sentinel_real_v4_balanced/sampled_scam_ids.json')
sh('cp sentinel/outputs/sentinel_real_v4_balanced_smoke/smoke_check.json sentinel/outputs/sentinel_real_v4_balanced/smoke_check.json && cp sentinel/outputs/sentinel_real_v4_balanced_smoke/training_log.json sentinel/outputs/sentinel_real_v4_balanced/smoke_training_log.json')
sh('cd sentinel/outputs && zip -qr sentinel_real_v4_balanced_results.zip sentinel_real_v4_balanced -x "sentinel_real_v4_balanced/lora/*" "sentinel_real_v4_balanced/checkpoint-*" "sentinel_real_v4_balanced/gguf/*" && unzip -tq sentinel_real_v4_balanced_results.zip && sha256sum sentinel_real_v4_balanced_results.zip')
from google.colab import files
files.download('sentinel/outputs/sentinel_real_v4_balanced_results.zip')